# Tutorial 11D: Inefficiency Determinants and Heteroscedastic Models

**Module**: Stochastic Frontier Analysis | **Difficulty**: Intermediate-Advanced | **Duration**: 3-4 hours

**Prerequisites**: Tutorials 01, 02; Marginal effects concepts; Delta method basics

## Learning Objectives

By the end of this tutorial, you will be able to:

1. **Model** inefficiency determinants using the BC95 framework
2. **Interpret** location effects (Z variables on mean inefficiency)
3. **Estimate** Wang (2002) heteroscedastic inefficiency models
4. **Calculate** and **interpret** marginal effects on inefficiency
5. **Distinguish** location vs scale effects
6. **Conduct** policy analysis using determinants
7. **Simulate** counterfactual scenarios
8. **Compare** BC95 vs Wang (2002) specifications

## Table of Contents

1. [Economic Motivation](#1-economic-motivation)
2. [Theoretical Framework](#2-theoretical-framework)
3. [Setup and Data Loading](#3-setup-and-data-loading)
4. [BC95: Inefficiency Determinants](#4-bc95-inefficiency-determinants)
5. [Marginal Effects Analysis](#5-marginal-effects-analysis)
6. [Visualization of Determinants](#6-visualization-of-determinants)
7. [Policy Simulations](#7-policy-simulations)
8. [Wang (2002): Heteroscedastic Inefficiency](#8-wang-2002-heteroscedastic-inefficiency)
9. [Comparing BC95 vs Wang (2002)](#9-comparing-bc95-vs-wang-2002)
10. [Counterfactual Analysis](#10-counterfactual-analysis)
11. [Exercises](#11-exercises)
12. [Summary](#12-summary)
13. [References](#13-references)

## 1. Economic Motivation

<a id="1-economic-motivation"></a>

### Why Does Inefficiency Vary?

In earlier tutorials, we estimated inefficiency as a single one-sided error term. But in practice, **inefficiency is not random** — it depends on observable factors:

**Observable Factors**:
- **Accreditation**: Hospitals meeting quality standards may be better managed
- **Teaching status**: Teaching hospitals juggle education and care
- **Occupancy rate**: Higher utilization may signal better resource management
- **Human capital**: Education, training, experience of workers
- **Market structure**: Competition, regulation, market concentration

**Policy Questions**:
1. Does accreditation improve hospital efficiency?
2. Do teaching hospitals operate differently from non-teaching ones?
3. How does occupancy rate relate to efficiency?
4. Which policies would be most cost-effective at improving efficiency?

### Two Types of Effects

| Effect Type | What It Changes | Example |
|-------------|----------------|----------|
| **Location (Mean)** | Shifts average inefficiency | Accreditation reduces mean inefficiency by 10% |
| **Scale (Variance)** | Changes dispersion of inefficiency | Competition increases variance (some improve, others struggle) |

The **BC95 model** captures location effects. The **Wang (2002) model** captures both location and scale effects.

## 2. Theoretical Framework

<a id="2-theoretical-framework"></a>

### 2.1 BC95: Inefficiency Determinants (Location Effects)

**Battese & Coelli (1995)** extended the SFA model to include determinants of inefficiency:

$$y_{it} = \mathbf{x}_{it}'\boldsymbol{\beta} + v_{it} - u_{it}$$

$$u_{it} \sim N^+(\mu_{it}, \sigma^2_u)$$

$$\mu_{it} = \mathbf{z}_{it}'\boldsymbol{\delta}$$

where $\mathbf{z}_{it}$ are **inefficiency determinants** and $\boldsymbol{\delta}$ are parameters to estimate.

**Interpretation of $\delta_k$**:
- $\delta_k > 0$: Variable $z_k$ **increases** mean inefficiency (detrimental)
- $\delta_k < 0$: Variable $z_k$ **decreases** mean inefficiency (beneficial)

**Key advantage**: The frontier and inefficiency equations are estimated **simultaneously** by MLE (one-step approach), avoiding the bias of two-step procedures.

### 2.2 Wang (2002): Heteroscedastic Inefficiency

**Wang (2002)** extended BC95 by allowing the **variance** of inefficiency to also depend on observables:

$$u_i \sim N^+(\mu_i, \sigma^2_{u,i})$$

$$\mu_i = \mathbf{z}_i'\boldsymbol{\delta} \quad \text{(location determinants)}$$

$$\ln(\sigma^2_{u,i}) = \mathbf{w}_i'\boldsymbol{\gamma} \quad \text{(scale determinants)}$$

**Two sets of determinants**:
- **Z → $\mu$** (location): Shift the mean of inefficiency
- **W → $\sigma^2_u$** (scale): Change the dispersion of inefficiency

### 2.3 Marginal Effects

The marginal effect of a determinant $z_k$ on expected inefficiency is **not simply $\delta_k$** because $E[u]$ is a nonlinear function of $\mu$ and $\sigma_u$:

$$\frac{\partial E[u_{it}]}{\partial z_k} = \delta_k \cdot \left[1 - \frac{\lambda \phi(\lambda)}{\Phi(\lambda)}\right]$$

where $\lambda = \mu/\sigma_u$, $\phi$ is the standard normal PDF, and $\Phi$ is the CDF.

Standard errors for marginal effects are computed via the **delta method**.

## 3. Setup and Data Loading

<a id="3-setup-and-data-loading"></a>

In [ ]:
# ============================================================
# Section 3: Setup and Data Loading
# ============================================================

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
from pathlib import Path

from scipy import stats

from panelbox.frontier import StochasticFrontier
from panelbox.frontier.utils.marginal_effects import (
    marginal_effects_bc95,
    marginal_effects_summary,
    marginal_effects_wang_2002,
)

np.random.seed(42)

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("husl")

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
FIGURES_DIR = BASE_DIR / "outputs" / "figures" / "04_determinants"
TABLES_DIR_LATEX = BASE_DIR / "outputs" / "tables" / "latex"
TABLES_DIR_HTML = BASE_DIR / "outputs" / "tables" / "html"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR_LATEX.mkdir(parents=True, exist_ok=True)
TABLES_DIR_HTML.mkdir(parents=True, exist_ok=True)

print("Setup complete!")

In [ ]:
# Load the hospital panel dataset
data = pd.read_csv(DATA_DIR / "hospital_panel.csv")

print(f"Dataset shape: {data.shape}")
print(f"Variables: {data.columns.tolist()}")
print(f"\nNumber of hospitals: {data['hospital_id'].nunique()}")
print(f"Number of years: {data['year'].nunique()}")
print(f"Year range: {data['year'].min()} - {data['year'].max()}")
print("\nFirst 5 observations:")
display(data.head())

print("\nSummary Statistics:")
display(data.describe().round(4))

### Understanding the Variables

| Variable | Description | Type | Role |
|----------|------------|------|------|
| `hospital_id` | Hospital identifier | ID | Panel entity |
| `year` | Year | ID | Panel time |
| `log_output` | Log of cases treated | Continuous | **Dependent variable** |
| `log_labor` | Log of medical staff | Continuous | **Input** |
| `log_capital` | Log of capital stock | Continuous | **Input** |
| `log_supplies` | Log of supplies/materials | Continuous | **Input** |
| `teaching` | Teaching hospital (1=yes) | Binary | **Z variable** (location) |
| `accreditation` | Accredited (1=yes) | Binary | **Z variable** (location) |
| `occupancy_rate` | Bed occupancy rate | Continuous | **Z variable** (location) |
| `avg_stay` | Average length of stay | Continuous | **W variable** (scale) |

## 4. BC95: Inefficiency Determinants

<a id="4-bc95-inefficiency-determinants"></a>

We now estimate the Battese & Coelli (1995) model with inefficiency determinants. The key idea is that the **mean** of the truncated normal inefficiency distribution depends on observable hospital characteristics.

**Model specification**:
- **Frontier**: $\ln(\text{output}_{it}) = \beta_0 + \beta_1 \ln(\text{labor}) + \beta_2 \ln(\text{capital}) + \beta_3 \ln(\text{supplies}) + v_{it} - u_{it}$
- **Inefficiency mean**: $\mu_{it} = \delta_0 + \delta_1 \cdot \text{teaching} + \delta_2 \cdot \text{accreditation} + \delta_3 \cdot \text{occupancy\_rate}$
- **Distribution**: $u_{it} \sim N^+(\mu_{it}, \sigma^2_u)$

In [ ]:
# Estimate BC95 model with inefficiency determinants
model_bc95 = StochasticFrontier(
    data=data,
    depvar="log_output",
    exog=["log_labor", "log_capital", "log_supplies"],
    inefficiency_vars=["teaching", "accreditation", "occupancy_rate"],
    entity="hospital_id",
    time="year",
    frontier="production",
    dist="truncated_normal",
    model_type="bc95",
)

result_bc95 = model_bc95.fit(method="mle", optimizer="L-BFGS-B")

print("=" * 70)
print("BC95: INEFFICIENCY DETERMINANTS MODEL")
print("=" * 70)
print(result_bc95.summary())

In [ ]:
# Display frontier parameters
print("=" * 60)
print("FRONTIER PARAMETERS (Technology)")
print("=" * 60)

frontier_vars = ["const", "log_labor", "log_capital", "log_supplies"]
for var in frontier_vars:
    if var in result_bc95.params.index:
        coef = result_bc95.params[var]
        se = result_bc95.se[var]
        tval = result_bc95.tvalues[var]
        pval = result_bc95.pvalues[var]
        sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
        print(f"  {var:<15}: {coef:>8.4f}  (SE: {se:.4f}, t: {tval:.2f})  {sig}")

# Display inefficiency determinants (delta parameters)
print(f"\n{'=' * 60}")
print("INEFFICIENCY DETERMINANTS (delta parameters)")
print("=" * 60)
print("Positive delta -> INCREASES inefficiency (bad)")
print("Negative delta -> DECREASES inefficiency (good)")
print()

delta_vars = [p for p in result_bc95.params.index if p.startswith("delta_")]
for var in delta_vars:
    coef = result_bc95.params[var]
    se = result_bc95.se[var]
    pval = result_bc95.pvalues[var]
    sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
    direction = "INCREASES ineff." if coef > 0 else "DECREASES ineff."
    print(f"  {var:<25}: delta = {coef:>8.4f}  (SE: {se:.4f}, p={pval:.4f})  {sig}  [{direction}]")

# Variance parameters
print(f"\n{'=' * 60}")
print("VARIANCE PARAMETERS")
print("=" * 60)
print(f"  sigma_v:  {result_bc95.sigma_v:.4f}")
print(f"  sigma_u:  {result_bc95.sigma_u:.4f}")
print(f"  lambda:   {result_bc95.lambda_param:.4f}")
print(f"  gamma:    {result_bc95.gamma:.4f}")

### Interpreting the Delta Parameters

The estimated $\delta$ parameters tell us how each determinant affects the **mean** of the truncated normal inefficiency distribution:

- **Teaching ($\delta_1$)**: If negative, teaching hospitals have lower mean inefficiency, suggesting their dual mission does not compromise operational efficiency.
- **Accreditation ($\delta_2$)**: If negative, accredited hospitals are more efficient on average, consistent with the idea that quality standards improve management.
- **Occupancy rate ($\delta_3$)**: If negative, higher occupancy is associated with lower inefficiency, suggesting better resource utilization.

**Important**: The raw $\delta$ coefficients do not directly give the marginal effect on $E[u]$ because the expectation of a truncated normal is a nonlinear function of $\mu$ and $\sigma_u$. We compute proper marginal effects in the next section.

## 5. Marginal Effects Analysis

<a id="5-marginal-effects-analysis"></a>

The marginal effect of a determinant $z_k$ on expected inefficiency is:

$$\frac{\partial E[u]}{\partial z_k} = \delta_k \cdot \left[1 - \frac{\lambda \phi(\lambda)}{\Phi(\lambda)}\right]$$

This scaling factor accounts for the truncation. Standard errors are computed via the delta method.

In [ ]:
# Calculate marginal effects using the result object's method
try:
    me_bc95 = result_bc95.marginal_effects(method="location")
    print("=" * 80)
    print("MARGINAL EFFECTS ON EXPECTED INEFFICIENCY (BC95)")
    print("=" * 80)
    display(me_bc95)
except Exception as e:
    print(f"marginal_effects via result object failed: {e}")
    # Fallback to standalone function
    me_bc95 = marginal_effects_bc95(result_bc95)
    print("=" * 80)
    print("MARGINAL EFFECTS ON EXPECTED INEFFICIENCY (BC95)")
    print("=" * 80)
    display(me_bc95)

# Filter out constant term for subsequent analysis
me_bc95 = me_bc95[me_bc95["variable"] != "const"].reset_index(drop=True)

# Compute confidence intervals from std_error
me_bc95["ci_lower"] = me_bc95["marginal_effect"] - 1.96 * me_bc95["std_error"]
me_bc95["ci_upper"] = me_bc95["marginal_effect"] + 1.96 * me_bc95["std_error"]

print("\nMarginal effects (excluding constant, with 95% CI):")
display(me_bc95[["variable", "marginal_effect", "std_error", "ci_lower", "ci_upper", "p_value"]])

print("\nInterpretation:")
print("  ME > 0 -> One-unit increase in variable INCREASES expected inefficiency")
print("  ME < 0 -> One-unit increase in variable DECREASES expected inefficiency")

In [ ]:
# Formatted summary of marginal effects
try:
    summary_text = marginal_effects_summary(me_bc95)
    print(summary_text)
except Exception as e:
    print(f"Summary formatting: {e}")

# Manual detailed interpretation
print("\nDetailed Interpretation:")
for _, row in me_bc95.iterrows():
    var = row["variable"]
    me = row["marginal_effect"]
    pval = row["p_value"]
    if np.isnan(pval):
        sig = "n.s."
    else:
        sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else "n.s."))
    if not np.isnan(pval) and pval < 0.05:
        if me > 0:
            print(f"  {var}: ME = {me:.4f} ({sig})")
            print(f"    -> One-unit increase RAISES expected inefficiency by {me:.4f}")
            print(f"    -> Policy: Reduce {var} to improve efficiency")
        else:
            print(f"  {var}: ME = {me:.4f} ({sig})")
            print(f"    -> One-unit increase LOWERS expected inefficiency by {abs(me):.4f}")
            print(f"    -> Policy: Increase {var} to improve efficiency")
    else:
        print(f"  {var}: ME = {me:.4f} (not significant, p={pval:.3f})")

In [ ]:
# Visualize marginal effects with confidence intervals
fig, ax = plt.subplots(figsize=(10, 6))

variables = me_bc95["variable"].values
effects = me_bc95["marginal_effect"].values
std_errors = me_bc95["std_error"].values
ci_lower = me_bc95["ci_lower"].values
ci_upper = me_bc95["ci_upper"].values

colors = ["#e74c3c" if e > 0 else "#27ae60" for e in effects]

y_pos = np.arange(len(variables))
ax.barh(y_pos, effects, color=colors, alpha=0.7, edgecolor="black", linewidth=0.5, height=0.5)

# Add error bars only if std_errors are finite and > 0
has_valid_se = np.isfinite(std_errors) & (std_errors > 0)
if has_valid_se.any():
    xerr_lower = np.where(has_valid_se, effects - ci_lower, 0)
    xerr_upper = np.where(has_valid_se, ci_upper - effects, 0)
    ax.errorbar(
        effects,
        y_pos,
        xerr=[xerr_lower, xerr_upper],
        fmt="none",
        ecolor="black",
        capsize=5,
        linewidth=1.5,
    )

ax.axvline(0, color="black", linewidth=1, linestyle="-")
ax.set_yticks(y_pos)
ax.set_yticklabels(variables, fontsize=12)
ax.set_xlabel("Marginal Effect on E[u] (Expected Inefficiency)", fontsize=13)
ax.set_title(
    "BC95: Marginal Effects on Inefficiency\n(with 95% Confidence Intervals)",
    fontsize=14,
    fontweight="bold",
)

# Add significance stars
for i, (_, row) in enumerate(me_bc95.iterrows()):
    pval = row["p_value"]
    if np.isnan(pval):
        sig = ""
    else:
        sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
    if sig:
        x_pos = (
            max(abs(effects[i]), abs(ci_upper[i]) if np.isfinite(ci_upper[i]) else abs(effects[i]))
            * 1.15
        )
        if effects[i] < 0:
            x_pos = -x_pos
        ax.text(x_pos, i, sig, va="center", ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "bc95_marginal_effects.png", dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved: bc95_marginal_effects.png")

## 6. Visualization of Determinants

<a id="6-visualization-of-determinants"></a>

Let's visualize how efficiency varies across the determinant groups.

In [ ]:
# Get efficiency estimates
eff_bc95 = result_bc95.efficiency(estimator="bc", ci_level=0.95)
data_with_eff = data.copy()
data_with_eff["te_bc95"] = eff_bc95["efficiency"].values

print("Efficiency summary:")
display(data_with_eff["te_bc95"].describe().round(4))

In [ ]:
# Efficiency by teaching status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot by teaching status
teaching_labels = {0: "Non-Teaching", 1: "Teaching"}
data_with_eff["teaching_label"] = data_with_eff["teaching"].map(teaching_labels)

sns.boxplot(
    x="teaching_label",
    y="te_bc95",
    data=data_with_eff,
    ax=axes[0],
    palette=["#FF9800", "#2196F3"],
    width=0.5,
)
axes[0].set_xlabel("Hospital Type", fontsize=12)
axes[0].set_ylabel("Technical Efficiency (BC)", fontsize=12)
axes[0].set_title("Efficiency by Teaching Status", fontsize=13, fontweight="bold")

# Add Mann-Whitney test
te_teach0 = data_with_eff[data_with_eff["teaching"] == 0]["te_bc95"]
te_teach1 = data_with_eff[data_with_eff["teaching"] == 1]["te_bc95"]
mw_stat, mw_pval = stats.mannwhitneyu(te_teach0, te_teach1, alternative="two-sided")
axes[0].annotate(
    f"Mann-Whitney p = {mw_pval:.4f}",
    xy=(0.5, 0.02),
    xycoords="axes fraction",
    ha="center",
    fontsize=10,
    bbox={"boxstyle": "round", "facecolor": "lightyellow", "alpha": 0.8},
)

# Time series by teaching status
for teach in [0, 1]:
    subset = data_with_eff[data_with_eff["teaching"] == teach]
    avg_eff = subset.groupby("year")["te_bc95"].mean()
    label = "Teaching" if teach == 1 else "Non-Teaching"
    axes[1].plot(avg_eff.index, avg_eff.values, marker="o", label=label, linewidth=2)

axes[1].set_xlabel("Year", fontsize=12)
axes[1].set_ylabel("Average Technical Efficiency", fontsize=12)
axes[1].set_title("Efficiency Evolution by Teaching Status", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficiency_by_teaching.png", dpi=300, bbox_inches="tight")
plt.show()

# Summary statistics by group
print("\nEfficiency by Teaching Status:")
display(data_with_eff.groupby("teaching_label")["te_bc95"].describe().round(4))

In [ ]:
# Efficiency by accreditation status
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

accred_labels = {0: "Not Accredited", 1: "Accredited"}
data_with_eff["accred_label"] = data_with_eff["accreditation"].map(accred_labels)

sns.boxplot(
    x="accred_label",
    y="te_bc95",
    data=data_with_eff,
    ax=axes[0],
    palette=["#e74c3c", "#27ae60"],
    width=0.5,
)
axes[0].set_xlabel("Accreditation Status", fontsize=12)
axes[0].set_ylabel("Technical Efficiency (BC)", fontsize=12)
axes[0].set_title("Efficiency by Accreditation", fontsize=13, fontweight="bold")

# Efficiency vs occupancy rate scatter
axes[1].scatter(
    data_with_eff["occupancy_rate"],
    data_with_eff["te_bc95"],
    alpha=0.3,
    s=20,
    color="steelblue",
    edgecolors="navy",
    linewidth=0.2,
)

# Add regression line
z = np.polyfit(data_with_eff["occupancy_rate"], data_with_eff["te_bc95"], 1)
p = np.poly1d(z)
x_range = np.linspace(
    data_with_eff["occupancy_rate"].min(), data_with_eff["occupancy_rate"].max(), 100
)
axes[1].plot(x_range, p(x_range), "r-", linewidth=2, label="Linear fit")

corr = data_with_eff["occupancy_rate"].corr(data_with_eff["te_bc95"])
axes[1].annotate(
    f"Correlation: {corr:.3f}",
    xy=(0.05, 0.95),
    xycoords="axes fraction",
    fontsize=11,
    verticalalignment="top",
    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
)

axes[1].set_xlabel("Occupancy Rate", fontsize=12)
axes[1].set_ylabel("Technical Efficiency (BC)", fontsize=12)
axes[1].set_title("Efficiency vs Occupancy Rate", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "efficiency_by_size.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Policy Simulations

<a id="7-policy-simulations"></a>

Using the estimated marginal effects, we can simulate the impact of policy interventions on hospital efficiency. These counterfactual scenarios answer "what if" questions.

In [ ]:
# Policy simulation: Effect of universal accreditation
print("=" * 60)
print("POLICY SIMULATION: Universal Accreditation")
print("=" * 60)

# Current status
pct_accredited = data_with_eff["accreditation"].mean() * 100
print(f"\nCurrent accreditation rate: {pct_accredited:.1f}%")

# Non-accredited hospitals
non_accred = data_with_eff[data_with_eff["accreditation"] == 0]
accred = data_with_eff[data_with_eff["accreditation"] == 1]

print(f"Non-accredited hospitals: Mean TE = {non_accred['te_bc95'].mean():.4f}")
print(f"Accredited hospitals:     Mean TE = {accred['te_bc95'].mean():.4f}")
print(f"Efficiency gap:           {(accred['te_bc95'].mean() - non_accred['te_bc95'].mean()):.4f}")

# Marginal effect of accreditation (from ME analysis)
me_accred_row = me_bc95[me_bc95["variable"].str.contains("accreditation")]
if len(me_accred_row) > 0:
    me_accred = me_accred_row["marginal_effect"].values[0]
    print(f"\nMarginal Effect of accreditation on E[u]: {me_accred:.4f}")

    # Simulation: if all non-accredited hospitals become accredited
    current_mean_te = data_with_eff["te_bc95"].mean()
    current_mean_u = -np.log(current_mean_te)

    # For non-accredited hospitals, going from 0 to 1
    # Change in expected u = ME * (1 - 0) = ME
    n_non_accred = len(non_accred)
    n_total = len(data_with_eff)

    # Overall effect: weighted by proportion of non-accredited
    overall_change_u = me_accred * (n_non_accred / n_total)
    simulated_mean_te = np.exp(-(current_mean_u + overall_change_u))

    print(f"\nCurrent Mean TE:        {current_mean_te:.4f}")
    print(f"Simulated Mean TE:      {simulated_mean_te:.4f}")
    print(f"Efficiency Gain:        {(simulated_mean_te - current_mean_te):.4f}")
    pct_gain = 100 * (simulated_mean_te / current_mean_te - 1)
    print(f"Percentage Improvement: {pct_gain:.2f}%")
else:
    print("Accreditation not found in marginal effects")

In [ ]:
# Multiple policy scenarios
print("=" * 70)
print("MULTIPLE POLICY SCENARIOS")
print("=" * 70)

baseline_te = data_with_eff["te_bc95"].mean()
baseline_u = -np.log(baseline_te)

scenarios = [
    {"name": "Baseline (status quo)", "delta_u": 0},
]

# Build scenarios from ME results
for _, row in me_bc95.iterrows():
    var = row["variable"]
    me = row["marginal_effect"]
    # For binary variables: simulate changing from 0 to 1
    if var in ["teaching", "accreditation"]:
        clean_var = var.replace("delta_", "")
        current_rate = data_with_eff[clean_var].mean() if clean_var in data_with_eff.columns else 0
        delta = me * (1 - current_rate)  # Weighted by those not yet treated
        scenarios.append({"name": f"Universal {clean_var.title()}", "delta_u": delta})
    elif var in ["occupancy_rate"]:
        clean_var = var.replace("delta_", "")
        # Simulate 10% increase in occupancy
        scenarios.append({"name": f"10% increase in {clean_var}", "delta_u": me * 0.10})

# Combined scenario
combined_delta = sum(s["delta_u"] for s in scenarios[1:])
scenarios.append({"name": "All Policies Combined", "delta_u": combined_delta})

results_table = []
for scenario in scenarios:
    sim_u = baseline_u + scenario["delta_u"]
    sim_te = np.exp(-max(sim_u, 0))
    results_table.append(
        {
            "Scenario": scenario["name"],
            "Mean TE": sim_te,
            "Gain vs Baseline": sim_te - baseline_te,
            "Pct Gain (%)": 100 * (sim_te / baseline_te - 1),
        }
    )

results_df = pd.DataFrame(results_table)
display(results_df.round(4))

In [ ]:
# Visualize policy scenarios
fig, ax = plt.subplots(figsize=(10, 6))

scenario_names = results_df["Scenario"].values
gains = results_df["Pct Gain (%)"].values

colors = ["#95a5a6"] + ["#27ae60" if g > 0 else "#e74c3c" for g in gains[1:]]
bars = ax.barh(
    range(len(scenario_names)),
    gains,
    color=colors,
    alpha=0.8,
    edgecolor="black",
    linewidth=0.5,
    height=0.6,
)

ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(range(len(scenario_names)))
ax.set_yticklabels(scenario_names, fontsize=11)
ax.set_xlabel("Efficiency Gain (%)", fontsize=13)
ax.set_title("Policy Scenario Analysis: Impact on Mean Efficiency", fontsize=14, fontweight="bold")

# Add value labels
for i, (_bar, val) in enumerate(zip(bars, gains)):
    if val != 0:
        ax.text(
            val + 0.1 if val > 0 else val - 0.1,
            i,
            f"{val:+.2f}%",
            va="center",
            ha="left" if val > 0 else "right",
            fontsize=10,
        )

plt.tight_layout()
plt.savefig(FIGURES_DIR / "policy_scenarios.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Wang (2002): Heteroscedastic Inefficiency

<a id="8-wang-2002-heteroscedastic-inefficiency"></a>

The Wang (2002) model extends BC95 by allowing the **variance** of inefficiency to vary across observations. This captures the idea that some factors may not change the average level of inefficiency, but change its **dispersion**.

For example, average length of stay (`avg_stay`) could affect how variable hospital inefficiency is: hospitals with very long or very short stays may have more unpredictable efficiency.

**Model**:
- **Location**: $\mu_{it} = \delta_0 + \delta_1 \cdot \text{teaching} + \delta_2 \cdot \text{accreditation}$ (Z variables)
- **Scale**: $\ln(\sigma^2_{u,it}) = \gamma_0 + \gamma_1 \cdot \text{occupancy\_rate} + \gamma_2 \cdot \text{avg\_stay}$ (W variables)

In [ ]:
# Estimate Wang (2002) model with both location and scale determinants
model_wang = StochasticFrontier(
    data=data,
    depvar="log_output",
    exog=["log_labor", "log_capital", "log_supplies"],
    inefficiency_vars=["teaching", "accreditation"],  # Z: location (mean)
    het_vars=["occupancy_rate", "avg_stay"],  # W: scale (variance)
    entity="hospital_id",
    time="year",
    frontier="production",
    dist="truncated_normal",
)

result_wang = model_wang.fit(method="mle", optimizer="L-BFGS-B")

print("=" * 70)
print("WANG (2002): HETEROSCEDASTIC INEFFICIENCY MODEL")
print("=" * 70)
print(result_wang.summary())

In [ ]:
# Display location and scale determinants separately
print("=" * 60)
print("LOCATION DETERMINANTS (delta -> affect mean mu)")
print("=" * 60)

delta_vars_wang = [p for p in result_wang.params.index if p.startswith("delta_")]
for var in delta_vars_wang:
    coef = result_wang.params[var]
    se = result_wang.se[var]
    pval = result_wang.pvalues[var]
    sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
    print(f"  {var:<25}: delta = {coef:>8.4f}  (SE: {se:.4f})  {sig}")

print(f"\n{'=' * 60}")
print("SCALE DETERMINANTS (gamma -> affect variance sigma_u^2)")
print("=" * 60)

gamma_vars = [p for p in result_wang.params.index if p.startswith("gamma_")]
for var in gamma_vars:
    coef = result_wang.params[var]
    se = result_wang.se[var]
    pval = result_wang.pvalues[var]
    sig = "***" if pval < 0.01 else ("**" if pval < 0.05 else ("*" if pval < 0.1 else ""))
    direction = "INCREASES dispersion" if coef > 0 else "DECREASES dispersion"
    print(f"  {var:<25}: gamma = {coef:>8.4f}  (SE: {se:.4f})  {sig}  [{direction}]")

In [ ]:
# Marginal effects for Wang (2002)
print("=" * 70)
print("MARGINAL EFFECTS: WANG (2002)")
print("=" * 70)

# Location marginal effects
print("\nLocation Effects (dE[u]/dz):")
try:
    me_wang_loc = result_wang.marginal_effects(method="location")
    display(me_wang_loc)
except Exception as e:
    print(f"  Location ME error: {e}")
    me_wang_loc = marginal_effects_wang_2002(result_wang, method="mean")
    display(me_wang_loc)

# Scale marginal effects
print("\nScale Effects (dE[u]/dw):")
try:
    me_wang_scale = result_wang.marginal_effects(method="scale")
    display(me_wang_scale)
except Exception as e:
    print(f"  Scale ME error: {e}")
    try:
        me_wang_scale = marginal_effects_wang_2002(result_wang, method="variance")
        display(me_wang_scale)
    except Exception as e2:
        print(f"  Scale ME fallback error: {e2}")
        me_wang_scale = None

# Filter out constant and add CIs for both
if me_wang_loc is not None:
    me_wang_loc = me_wang_loc[me_wang_loc["variable"] != "const"].reset_index(drop=True)
    me_wang_loc["ci_lower"] = me_wang_loc["marginal_effect"] - 1.96 * me_wang_loc["std_error"]
    me_wang_loc["ci_upper"] = me_wang_loc["marginal_effect"] + 1.96 * me_wang_loc["std_error"]

if me_wang_scale is not None:
    me_wang_scale = me_wang_scale[me_wang_scale["variable"] != "const"].reset_index(drop=True)
    me_wang_scale["ci_lower"] = me_wang_scale["marginal_effect"] - 1.96 * me_wang_scale["std_error"]
    me_wang_scale["ci_upper"] = me_wang_scale["marginal_effect"] + 1.96 * me_wang_scale["std_error"]

print("\nInterpretation:")
print("  Location ME: how Z variables shift MEAN inefficiency")
print("  Scale ME: how W variables change DISPERSION of inefficiency")

In [ ]:
# Visualize location vs scale effects
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Location effects
if me_wang_loc is not None and len(me_wang_loc) > 0:
    vars_loc = me_wang_loc["variable"].values
    effects_loc = me_wang_loc["marginal_effect"].values
    se_loc = me_wang_loc["std_error"].values

    colors_loc = ["#e74c3c" if e > 0 else "#27ae60" for e in effects_loc]
    axes[0].barh(
        range(len(vars_loc)),
        effects_loc,
        color=colors_loc,
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
        height=0.5,
    )

    # Add error bars only for valid SEs
    has_valid = np.isfinite(se_loc) & (se_loc > 0)
    if has_valid.any():
        xerr = np.where(has_valid, 1.96 * se_loc, 0)
        axes[0].errorbar(
            effects_loc, range(len(vars_loc)), xerr=xerr, fmt="none", ecolor="black", capsize=5
        )

    axes[0].axvline(0, color="black", linewidth=1)
    axes[0].set_yticks(range(len(vars_loc)))
    axes[0].set_yticklabels(vars_loc, fontsize=11)
    axes[0].set_xlabel("Marginal Effect on E[u]", fontsize=12)
    axes[0].set_title("Location Effects (on Mean)", fontsize=13, fontweight="bold")

# Scale effects
if me_wang_scale is not None and len(me_wang_scale) > 0:
    vars_scale = me_wang_scale["variable"].values
    effects_scale = me_wang_scale["marginal_effect"].values
    se_scale = me_wang_scale["std_error"].values

    # Handle NaN effects
    effects_scale_plot = np.nan_to_num(effects_scale, nan=0.0)

    colors_scale = ["#e74c3c" if e > 0 else "#27ae60" for e in effects_scale_plot]
    axes[1].barh(
        range(len(vars_scale)),
        effects_scale_plot,
        color=colors_scale,
        alpha=0.7,
        edgecolor="black",
        linewidth=0.5,
        height=0.5,
    )

    has_valid = np.isfinite(se_scale) & (se_scale > 0) & np.isfinite(effects_scale)
    if has_valid.any():
        xerr = np.where(has_valid, 1.96 * se_scale, 0)
        axes[1].errorbar(
            effects_scale_plot,
            range(len(vars_scale)),
            xerr=xerr,
            fmt="none",
            ecolor="black",
            capsize=5,
        )

    axes[1].axvline(0, color="black", linewidth=1)
    axes[1].set_yticks(range(len(vars_scale)))
    axes[1].set_yticklabels(vars_scale, fontsize=11)
    axes[1].set_xlabel("Marginal Effect on sigma_u", fontsize=12)
    axes[1].set_title("Scale Effects (on Dispersion)", fontsize=13, fontweight="bold")
else:
    axes[1].text(
        0.5,
        0.5,
        "Scale effects not available",
        ha="center",
        va="center",
        fontsize=12,
        transform=axes[1].transAxes,
    )
    axes[1].set_title("Scale Effects (on Dispersion)", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "wang_location_vs_scale.png", dpi=300, bbox_inches="tight")
plt.show()

## 9. Comparing BC95 vs Wang (2002)

<a id="9-comparing-bc95-vs-wang-2002"></a>

We now formally compare the two models:

1. **Information criteria** (AIC, BIC): penalize for complexity
2. **Likelihood ratio test**: tests whether the scale effects in Wang (2002) are significant
3. **Efficiency comparison**: how do efficiency estimates differ?

In [ ]:
# Model comparison table
eff_bc95_mean = eff_bc95["efficiency"].mean()
eff_wang = result_wang.efficiency(estimator="bc", ci_level=0.95)
eff_wang_mean = np.nanmean(eff_wang["efficiency"].values)

comparison = pd.DataFrame(
    {
        "Model": ["BC95", "Wang (2002)"],
        "Log-Likelihood": [result_bc95.loglik, result_wang.loglik],
        "AIC": [result_bc95.aic, result_wang.aic],
        "BIC": [result_bc95.bic, result_wang.bic],
        "N Parameters": [result_bc95.nparams, result_wang.nparams],
        "Mean TE": [eff_bc95_mean, eff_wang_mean],
    }
)

print("=" * 70)
print("MODEL COMPARISON: BC95 vs WANG (2002)")
print("=" * 70)
display(comparison.round(4))

# Identify better model
better_aic = "Wang (2002)" if result_wang.aic < result_bc95.aic else "BC95"
better_bic = "Wang (2002)" if result_wang.bic < result_bc95.bic else "BC95"
print(f"\nPreferred by AIC: {better_aic}")
print(f"Preferred by BIC: {better_bic}")

In [ ]:
# Likelihood ratio test: BC95 vs Wang (2002)
# BC95 is restricted (no scale effects), Wang is unrestricted
# We need to be careful: models must be properly nested
# BC95 has: teaching, accreditation, occupancy_rate as Z vars
# Wang has: teaching, accreditation as Z vars + occupancy_rate, avg_stay as W vars
# These are not directly nested in the standard sense.
# We use model comparison via AIC/BIC instead.

print("=" * 60)
print("MODEL SELECTION ANALYSIS")
print("=" * 60)

delta_aic = result_bc95.aic - result_wang.aic
delta_bic = result_bc95.bic - result_wang.bic

print(f"\nDelta AIC (BC95 - Wang): {delta_aic:.4f}")
if delta_aic > 2:
    print("  -> Strong evidence in favor of Wang (2002)")
elif delta_aic > 0:
    print("  -> Weak evidence in favor of Wang (2002)")
elif delta_aic > -2:
    print("  -> Weak evidence in favor of BC95")
else:
    print("  -> Strong evidence in favor of BC95")

print(f"\nDelta BIC (BC95 - Wang): {delta_bic:.4f}")
if delta_bic > 6:
    print("  -> Very strong evidence in favor of Wang (2002)")
elif delta_bic > 2:
    print("  -> Positive evidence in favor of Wang (2002)")
elif delta_bic > -2:
    print("  -> Models are roughly equivalent")
else:
    print("  -> Evidence in favor of BC95")

print("\nConclusion:")
if delta_aic > 2 and delta_bic > 2:
    print("  Both AIC and BIC favor Wang (2002).")
    print("  -> Heteroscedasticity in inefficiency is important.")
    print("  -> The variance of inefficiency varies across hospitals.")
elif delta_aic < -2 and delta_bic < -2:
    print("  Both AIC and BIC favor BC95.")
    print("  -> The simpler model is adequate.")
    print("  -> No strong evidence for heteroscedastic inefficiency.")
else:
    print("  Mixed results: AIC and BIC give different recommendations.")
    print("  Consider the theoretical motivation for each model.")

In [ ]:
# Efficiency comparison scatter plot
eff_bc95_vals = eff_bc95["efficiency"].values
eff_wang_vals = eff_wang["efficiency"].values

# Filter out NaN values for plotting and correlation
valid_mask = np.isfinite(eff_bc95_vals) & np.isfinite(eff_wang_vals)
bc95_valid = eff_bc95_vals[valid_mask]
wang_valid = eff_wang_vals[valid_mask]

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(
    bc95_valid, wang_valid, alpha=0.3, s=20, color="steelblue", edgecolors="navy", linewidth=0.2
)
ax.plot([0, 1], [0, 1], "r--", linewidth=2, label="45-degree line")

if len(bc95_valid) > 2:
    corr = np.corrcoef(bc95_valid, wang_valid)[0, 1]
    rho, _ = stats.spearmanr(bc95_valid, wang_valid)
    ax.text(
        0.05,
        0.95,
        f"Pearson r = {corr:.3f}\nSpearman rho = {rho:.3f}",
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment="top",
        bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
    )

ax.set_xlabel("Efficiency (BC95)", fontsize=13)
ax.set_ylabel("Efficiency (Wang 2002)", fontsize=13)
ax.set_title(
    "Comparison of Efficiency Estimates:\nBC95 vs Wang (2002)", fontsize=14, fontweight="bold"
)
ax.legend(fontsize=12)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "bc95_vs_wang.png", dpi=300, bbox_inches="tight")
plt.show()

n_nan = (~valid_mask).sum()
if n_nan > 0:
    print(f"Note: {n_nan} observations excluded due to NaN efficiency values")
print(f"Mean efficiency difference (Wang - BC95): {np.nanmean(eff_wang_vals - eff_bc95_vals):.4f}")
print(f"Max absolute difference: {np.nanmax(np.abs(eff_wang_vals - eff_bc95_vals)):.4f}")

## 10. Counterfactual Analysis

<a id="10-counterfactual-analysis"></a>

Counterfactual analysis asks: "What would efficiency look like if we changed certain conditions?" This is the bridge between econometric analysis and policy recommendations.

In [ ]:
# Counterfactual: Effect of changing accreditation and teaching status
print("=" * 70)
print("COUNTERFACTUAL ANALYSIS")
print("=" * 70)

# Group analysis
data_with_eff["te_wang"] = eff_wang["efficiency"].values

# Compare across both teaching and accreditation
groups = (
    data_with_eff.groupby(["teaching", "accreditation"])
    .agg(
        mean_te_bc95=("te_bc95", "mean"),
        mean_te_wang=("te_wang", "mean"),
        count=("hospital_id", "count"),
        mean_occupancy=("occupancy_rate", "mean"),
        mean_avg_stay=("avg_stay", "mean"),
    )
    .round(4)
)

groups.index = groups.index.set_names(["Teaching", "Accredited"])
print("\nEfficiency by Teaching x Accreditation:")
display(groups)

# Most and least efficient groups
best_group = groups["mean_te_bc95"].idxmax()
worst_group = groups["mean_te_bc95"].idxmin()
print(
    f"\nMost efficient group:  Teaching={best_group[0]}, Accredited={best_group[1]}  "
    f"(TE={groups.loc[best_group, 'mean_te_bc95']:.4f})"
)
print(
    f"Least efficient group: Teaching={worst_group[0]}, Accredited={worst_group[1]}  "
    f"(TE={groups.loc[worst_group, 'mean_te_bc95']:.4f})"
)
gap = groups.loc[best_group, "mean_te_bc95"] - groups.loc[worst_group, "mean_te_bc95"]
print(f"Efficiency gap: {gap:.4f}")

In [ ]:
# Efficiency distribution comparison: BC95 vs Wang
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BC95 histogram
eff_bc95_clean = eff_bc95_vals[np.isfinite(eff_bc95_vals)]
axes[0].hist(
    eff_bc95_clean,
    bins=30,
    density=True,
    alpha=0.7,
    color="steelblue",
    edgecolor="black",
    linewidth=0.5,
)
if len(eff_bc95_clean) > 1:
    kde_bc = stats.gaussian_kde(eff_bc95_clean)
    x_range = np.linspace(eff_bc95_clean.min(), eff_bc95_clean.max(), 200)
    axes[0].plot(x_range, kde_bc(x_range), "r-", linewidth=2)
axes[0].axvline(
    eff_bc95_clean.mean(),
    color="green",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {eff_bc95_clean.mean():.3f}",
)
axes[0].set_xlabel("Technical Efficiency", fontsize=12)
axes[0].set_ylabel("Density", fontsize=12)
axes[0].set_title("BC95 Efficiency Distribution", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)

# Wang histogram (filter NaN values)
eff_wang_clean = eff_wang_vals[np.isfinite(eff_wang_vals)]
if len(eff_wang_clean) > 1:
    axes[1].hist(
        eff_wang_clean,
        bins=30,
        density=True,
        alpha=0.7,
        color="#FF9800",
        edgecolor="black",
        linewidth=0.5,
    )
    kde_wang = stats.gaussian_kde(eff_wang_clean)
    x_range2 = np.linspace(eff_wang_clean.min(), eff_wang_clean.max(), 200)
    axes[1].plot(x_range2, kde_wang(x_range2), "r-", linewidth=2)
    axes[1].axvline(
        eff_wang_clean.mean(),
        color="green",
        linestyle="--",
        linewidth=2,
        label=f"Mean: {eff_wang_clean.mean():.3f}",
    )
    axes[1].legend(fontsize=10)
else:
    axes[1].text(
        0.5,
        0.5,
        f"Wang efficiency has {np.isnan(eff_wang_vals).sum()} NaN values\nout of {len(eff_wang_vals)}",
        ha="center",
        va="center",
        fontsize=12,
        transform=axes[1].transAxes,
    )
axes[1].set_xlabel("Technical Efficiency", fontsize=12)
axes[1].set_ylabel("Density", fontsize=12)
axes[1].set_title("Wang (2002) Efficiency Distribution", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

n_nan = np.isnan(eff_wang_vals).sum()
if n_nan > 0:
    print(f"Note: {n_nan} Wang efficiency values are NaN ({100 * n_nan / len(eff_wang_vals):.1f}%)")
    print(f"Valid Wang efficiency: mean={np.nanmean(eff_wang_vals):.4f}")

## 11. Export Results

Export all results to LaTeX and HTML for reporting.

In [ ]:
# Export BC95 results
try:
    bc95_latex = result_bc95.to_latex(
        caption="BC95 Inefficiency Determinants Model", label="tab:bc95_results"
    )
    with open(TABLES_DIR_LATEX / "04_bc95_results.tex", "w") as f:
        f.write(bc95_latex)
    print("BC95 LaTeX table saved.")
except Exception as e:
    print(f"BC95 LaTeX: {e}")

try:
    bc95_html = result_bc95.to_html()
    with open(TABLES_DIR_HTML / "04_bc95_results.html", "w") as f:
        f.write(bc95_html)
    print("BC95 HTML table saved.")
except Exception as e:
    print(f"BC95 HTML: {e}")

# Export Wang results
try:
    wang_latex = result_wang.to_latex(
        caption="Wang (2002) Heteroscedastic Inefficiency Model", label="tab:wang_results"
    )
    with open(TABLES_DIR_LATEX / "04_wang_results.tex", "w") as f:
        f.write(wang_latex)
    print("Wang LaTeX table saved.")
except Exception as e:
    print(f"Wang LaTeX: {e}")

try:
    wang_html = result_wang.to_html()
    with open(TABLES_DIR_HTML / "04_wang_results.html", "w") as f:
        f.write(wang_html)
    print("Wang HTML table saved.")
except Exception as e:
    print(f"Wang HTML: {e}")

# Export marginal effects tables
me_bc95_latex = me_bc95.to_latex(
    index=False,
    float_format="%.4f",
    caption="Marginal Effects on Inefficiency (BC95)",
    label="tab:me_bc95",
)
with open(TABLES_DIR_LATEX / "04_marginal_effects_bc95.tex", "w") as f:
    f.write(me_bc95_latex)

me_bc95_html = me_bc95.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "04_marginal_effects_bc95.html", "w") as f:
    f.write(me_bc95_html)

# Export comparison table
comparison_latex = comparison.to_latex(
    index=False,
    float_format="%.4f",
    caption="BC95 vs Wang (2002) Model Comparison",
    label="tab:model_comparison",
)
with open(TABLES_DIR_LATEX / "04_model_comparison.tex", "w") as f:
    f.write(comparison_latex)

comparison_html = comparison.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "04_model_comparison.html", "w") as f:
    f.write(comparison_html)

# Export policy scenarios
policy_latex = results_df.to_latex(
    index=False,
    float_format="%.4f",
    caption="Policy Scenario Analysis",
    label="tab:policy_scenarios",
)
with open(TABLES_DIR_LATEX / "04_policy_scenarios.tex", "w") as f:
    f.write(policy_latex)

policy_html = results_df.to_html(index=False, float_format="%.4f")
with open(TABLES_DIR_HTML / "04_policy_scenarios.html", "w") as f:
    f.write(policy_html)

print("\nAll tables exported successfully!")

## 11. Exercises

<a id="11-exercises"></a>

These exercises use the `school_panel.csv` dataset to practice the concepts from this tutorial.

### Exercise 1: School Efficiency Determinants (Easy)

Load the `school_panel.csv` dataset and estimate a BC95 model to identify the determinants of school inefficiency.

**Tasks**:
1. Load the school data and explore its structure
2. Create a dummy variable for private schools (`school_type == 'private'`)
3. Estimate a BC95 model with:
   - **Frontier**: `log_output` ~ `log_teachers + log_budget + log_facilities`
   - **Z variables**: `teacher_experience`, `class_size`, `ses_index`
4. Calculate marginal effects
5. Identify which determinant has the largest impact on inefficiency

**Hints**:
- Use `inefficiency_vars=['teacher_experience', 'class_size', 'ses_index']`
- Remember: positive delta = increases inefficiency (bad)

In [ ]:
# Exercise 1: Your code here

# Step 1: Load data
# school_data = pd.read_csv(DATA_DIR / 'school_panel.csv')
# display(school_data.head())

# Step 2: Create private dummy
# school_data['private'] = (school_data['school_type'] == 'private').astype(int)

# Step 3: Estimate BC95
# model_school = StochasticFrontier(
#     data=school_data,
#     depvar='log_output',
#     exog=['log_teachers', 'log_budget', 'log_facilities'],
#     inefficiency_vars=['teacher_experience', 'class_size', 'ses_index'],
#     entity='school_id',
#     time='year',
#     frontier='production',
#     dist='truncated_normal',
#     model_type='bc95',
# )
# result_school = model_school.fit(method='mle', optimizer='L-BFGS-B')
# print(result_school.summary())

# Step 4: Marginal effects
# me_school = result_school.marginal_effects(method='location')
# display(me_school)

# Step 5: Identify most impactful determinant
# most_impactful = me_school.loc[me_school['marginal_effect'].abs().idxmax()]
# print(f'\nMost impactful: {most_impactful["variable"]} (ME = {most_impactful["marginal_effect"]:.4f})')

### Exercise 2: Heteroscedastic Analysis (Medium)

Estimate a Wang (2002) model for the school data to test whether the variance of inefficiency depends on school characteristics.

**Tasks**:
1. Estimate a Wang (2002) model with:
   - **Z variables** (location): `teacher_experience`, `ses_index`
   - **W variables** (scale): `class_size`, `private` (dummy for private schools)
2. Calculate both location and scale marginal effects
3. Compare with the BC95 model from Exercise 1 (AIC/BIC)
4. Interpret: Which factors shift the mean vs. the dispersion of inefficiency?

**Hints**:
- Use `het_vars=['class_size', 'private']` for scale determinants
- Compare AIC and BIC between BC95 and Wang

In [ ]:
# Exercise 2: Your code here

# Step 1: Estimate Wang (2002)
# model_wang_school = StochasticFrontier(
#     data=school_data,
#     depvar='log_output',
#     exog=['log_teachers', 'log_budget', 'log_facilities'],
#     inefficiency_vars=['teacher_experience', 'ses_index'],
#     het_vars=['class_size', 'private'],
#     entity='school_id',
#     time='year',
#     frontier='production',
#     dist='truncated_normal',
# )
# result_wang_school = model_wang_school.fit()
# print(result_wang_school.summary())

# Step 2: Marginal effects
# me_loc = result_wang_school.marginal_effects(method='location')
# me_scale = result_wang_school.marginal_effects(method='scale')
# print('Location ME:'); display(me_loc)
# print('Scale ME:'); display(me_scale)

# Step 3: Model comparison
# print(f'BC95  AIC: {result_school.aic:.4f}, BIC: {result_school.bic:.4f}')
# print(f'Wang  AIC: {result_wang_school.aic:.4f}, BIC: {result_wang_school.bic:.4f}')

### Exercise 3: Cost-Benefit Analysis (Hard)

Using the BC95 model from Exercise 1, conduct a policy cost-benefit analysis.

**Tasks**:
1. Use marginal effects to estimate the efficiency gain from:
   - Increasing teacher experience by 5 years (through professional development)
   - Reducing class size by 5 students
   - Increasing SES index by 0.1 (through community programs)
2. Assume hypothetical costs per unit change (you decide reasonable values)
3. Rank the three policies by cost-effectiveness (efficiency gain per dollar)
4. Discuss which policy you would recommend and why

**Hints**:
- Efficiency gain = ME * change_in_variable (approximately)
- Cost-effectiveness = efficiency_gain / cost
- Consider both statistical significance and practical significance

In [ ]:
# Exercise 3: Your code here

# Define policy scenarios with costs
# policies = [
#     {'name': 'Teacher Development (+5 yrs)', 'variable': 'teacher_experience',
#      'change': 5, 'cost_per_school': 50000},
#     {'name': 'Class Size Reduction (-5 students)', 'variable': 'class_size',
#      'change': -5, 'cost_per_school': 100000},
#     {'name': 'Community Programs (+0.1 SES)', 'variable': 'ses_index',
#      'change': 0.1, 'cost_per_school': 30000},
# ]

# for policy in policies:
#     var = policy['variable']
#     me_row = me_school[me_school['variable'].str.contains(var)]
#     if len(me_row) > 0:
#         me = me_row['marginal_effect'].values[0]
#         eff_change = -me * policy['change']  # Negative because ME is on u, not TE
#         cost_eff = eff_change / policy['cost_per_school'] * 1e6
#         print(f"{policy['name']}:")
#         print(f"  ME = {me:.4f}, Change in u = {me * policy['change']:.4f}")
#         print(f"  Approx TE gain = {eff_change:.4f}")
#         print(f"  Cost per school = ${policy['cost_per_school']:,}")
#         print(f"  Cost-effectiveness: {cost_eff:.4f} TE units per $1M\n")

## 12. Summary

<a id="12-summary"></a>

### Key Concepts

| Concept | Description |
|---------|-------------|
| **Inefficiency Determinants** | Observable variables that affect the level of inefficiency |
| **Location Effects** | Impact on the mean of inefficiency ($\mu$) |
| **Scale Effects** | Impact on the variance of inefficiency ($\sigma^2_u$) |
| **Marginal Effects** | $\partial E[u]/\partial z_k$ -- proper nonlinear effects |
| **Delta Method** | Standard error calculation for nonlinear functions |
| **BC95** | Location effects only (mean of truncated normal) |
| **Wang (2002)** | Both location and scale effects |
| **Policy Analysis** | Counterfactual simulations using marginal effects |

### Main Findings

1. **Accreditation** and **occupancy rate** are important determinants of hospital inefficiency
2. Marginal effects provide the correct interpretation (not raw delta coefficients)
3. The Wang (2002) model can reveal heteroscedastic patterns in inefficiency
4. Policy simulations translate econometric results into actionable recommendations
5. Model comparison (AIC/BIC) helps choose between BC95 and Wang specifications

### What's Next?

In Tutorial 05, we will cover:
- **Model testing and comparison**: Formal tests for model selection
- **Functional form tests**: Cobb-Douglas vs Translog
- **Sensitivity analysis**: Robustness of efficiency rankings

## 13. References

<a id="13-references"></a>

1. **Battese, G.E., & Coelli, T.J. (1995)**. "A model for technical inefficiency effects in a stochastic frontier production function for panel data." *Empirical Economics*, 20(2), 325-332.

2. **Wang, H.J. (2002)**. "Heteroscedasticity and non-monotonic efficiency effects of a stochastic frontier model." *Journal of Productivity Analysis*, 18(3), 241-253.

3. **Wang, H.J., & Schmidt, P. (2002)**. "One-step and two-step estimation of the effects of exogenous variables on technical efficiency levels." *Journal of Productivity Analysis*, 18(2), 129-144.

4. **Kumbhakar, S.C., & Lovell, C.A.K. (2000)**. *Stochastic Frontier Analysis*. Cambridge University Press.

5. **Coelli, T.J., Rao, D.S.P., O'Donnell, C.J., & Battese, G.E. (2005)**. *An Introduction to Efficiency and Productivity Analysis*. Springer, 2nd edition.

---

*This tutorial is part of the PanelBox Stochastic Frontier Analysis module. For questions or suggestions, please open an issue on the project repository.*